# Notebook 8c: Does markdown alone raise the score? (reverse test for RQ5)

**Before running:** a **T4 GPU** is faster (10–15 min), but if Colab has no GPU left, a normal CPU runtime works too (about 45–75 min). Notebooks 8 and 8b must have run first.

**Why this notebook.** Notebook 8b showed that most of E5's boost for AI-polished resumes (+0.66 of +0.74 SD) disappears when the markdown symbols (`**`, `*` bullets) are removed. That was tested by *taking markdown away* from the AI text. Here we test the other direction: *add* markdown to the **original** resumes, with **no rewording at all**. If E5 rises the same way, markdown is the cause, not the AI's wording.

**Versions of each original resume** (the same 181 resumes; letters and digits never change):
| version | what it looks like |
|---|---|
| `flat` | cleaned original, one block of text (the same as Notebook 8b's cleaned version) |
| `lines` | section headings on their own line with a colon (`Skills:`), one item or sentence per line. No symbols |
| `bold` | `lines` + bold headings (`**Skills:**`) |
| `bullets` | `lines` + a `* ` bullet before every item |
| `full` | bold headings + bullets, the way the AI rewrites looked |

**Also:** do the **job descriptions** contain markdown? If E5 likes `*` because job posts are full of `*`, the boost should be bigger for those jobs. And: if **every** resume had markdown, would ranking quality change?

**Run time:** about 10–15 minutes. No LLM in this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install -U sentence-transformers

import os, re, gc
import numpy as np
import pandas as pd
import torch
from scipy.stats import wilcoxon, spearmanr

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'   # no LLM here, so CPU works (about 45-75 min instead of 10-15)
print('running on', DEVICE)
DATA = '/content/drive/MyDrive/resume-job-matching/data'
RESULTS = '/content/drive/MyDrive/resume-job-matching/results'
SEED = 42
MAX_WORDS = 900

all_df = pd.concat([pd.read_parquet(f'{DATA}/{s}.parquet').assign(split=s) for s in ('train', 'val', 'test')],
                   ignore_index=True)
g = all_df.groupby('job_id').agg(candidates=('resume_id', 'size'),
                                 relevant=('relevance', lambda r: int((r > 0).sum())),
                                 levels=('relevance', 'nunique'))
usable = g[(g.candidates >= 5) & (g.relevant >= 1) & (g.levels >= 2)].index
evald = all_df[all_df.job_id.isin(usable)].reset_index(drop=True)
evald['pair_key'] = evald.job_id + '|' + evald.resume_id

def cut(text, n_words):
    w = text.split()
    return text if len(w) <= n_words else ' '.join(w[:n_words])
evald['resume_cut'] = evald.resume_text.map(lambda t: cut(t, MAX_WORDS))

nb8b = pd.read_parquet(f'{RESULTS}/08b_pair_scores.parquet')
ids = sorted(nb8b.resume_id.unique())
print(f'{len(usable)} usable jobs, {len(evald)} pairs, {len(ids)} resumes from Notebook 8b')
assert len(ids) == 181, 'Not the same resumes as Notebook 8b.'

## 1. Build the versions

Section headings are found where the scraper glued a heading word to the next word (`SkillsGAAP`, `Experience03/2017`), so only real headings are marked. Item boundaries are the other glued points (`AccountingAccount`) and sentence ends. The cleaning rules are the same as Notebook 8b.

In [ ]:
HEADINGS = ['Professional Summary', 'Executive Summary', 'Executive Profile', 'Career Overview', 'Professional Profile',
            'Professional Experience', 'Work History', 'Work Experience', 'Experience', 'Education and Training',
            'Education', 'Skill Highlights', 'Core Qualifications', 'Core Competencies', 'Qualifications', 'Skills',
            'Summary', 'Highlights', 'Accomplishments', 'Certifications', 'Additional Information', 'Languages',
            'Interests', 'Affiliations', 'Training', 'Projects', 'Publications', 'Awards', 'Objective']
HEAD = re.compile(r'(?<![A-Z])(' + '|'.join(HEADINGS) + r')(?=[A-Z0-9])')
H0, H1, BR = '\x01', '\x02', '\x00'   # heading start, heading end, item break

JOINED = re.compile(r'[a-z][A-Z]|[A-Z]{2}[a-z]{2}')
jobs_u = evald.drop_duplicates('job_id')
job_df = {}
for t in jobs_u.job_text:
    for w in set(re.findall(r'[A-Za-z]+', t)):
        if JOINED.search(w):
            job_df[w] = job_df.get(w, 0) + 1
KEEP = {w for w, n in job_df.items() if n >= 2}

def split_word(w):
    if w in KEEP or not JOINED.search(w):
        return w
    s = re.sub(r'^to(?=[A-Z])', 'to ', w)                        # 2017toCurrent is a date range, not an item break
    s = re.sub(r'([a-z])([A-Z])', r'\1' + BR + r'\2', s)        # a glued item boundary becomes an item break
    return re.sub(r'([A-Z]+)([A-Z][a-z]{2,})', r'\1' + BR + r'\2', s)

def structure(t):
    t = HEAD.sub(lambda m: H0 + m.group(1) + H1, t)
    t = re.sub(r'[A-Za-z]+', lambda m: split_word(m.group()), t)
    t = re.sub(r'(?<=[a-z]{4})(?=\d)', ' ', t)
    t = re.sub(r'(?<=\d)(?=to[A-Z ])|(?<=\d)(?=[A-Z][a-z])', ' ', t)
    t = re.sub(r'(?<=[a-z]{2}[.!?]) ?(?=[A-Z])', BR, t)            # sentence end: an item break
    t = re.sub(r'(?<=[a-z]{2}[;:])(?=[A-Z])', ' ', t)
    t = re.sub(r'(?<=[A-Za-z]{2}[,;])(?=[A-Za-z])', ' ', t)
    t = re.sub(r'\s*\|\s*', ' | ', t)
    return t

def render(t, style):
    head_on, head_off = ('**', ':**') if style in ('bold', 'full') else ('', ':')
    bullet = '* ' if style in ('bullets', 'full') else ''
    if style == 'flat':
        t = t.replace(H0, ' ').replace(H1, ' ').replace(BR, ' ')
        return re.sub(r'\s+', ' ', t).strip()
    t = t.replace(H0, '\n\n' + head_on).replace(H1, head_off + '\n').replace(BR, '\n')
    out = []
    for line in t.split('\n'):
        line = re.sub(r'[ \t]+', ' ', line).strip()
        if not line:
            if out and out[-1] != '':
                out.append('')
            continue
        is_head = line.startswith('**') and line.endswith(':**') or (style in ('lines', 'bullets') and line.endswith(':') and line[:-1] in HEADINGS)
        out.append(line if is_head else bullet + line)
    return '\n'.join(out).strip()

STYLES = ['flat', 'lines', 'bold', 'bullets', 'full']
res_cut = evald.drop_duplicates('resume_id').set_index('resume_id').resume_cut
struct = {r: structure(t) for r, t in res_cut.items()}
V = {s: {r: render(t, s) for r, t in struct.items()} for s in STYLES}

same = lambda a, b: re.sub(r'\W', '', a) == re.sub(r'\W', '', b)
for s in STYLES:
    bad = [r for r in ids if not same(V[s][r], res_cut[r])]
    assert not bad, f'{s}: content changed for {len(bad)} resumes.'
print('check passed: every version has exactly the same letters and digits as the original')
n_head = pd.Series({r: t.count(H0) for r, t in struct.items()})[ids]
n_items = pd.Series({r: t.count(BR) for r, t in struct.items()})[ids]
print('headings found per resume:', n_head.describe().round(1).to_dict())
print('item breaks per resume:', n_items.describe().round(1).to_dict())
print('resumes with no heading found:', int((n_head == 0).sum()))
print('\n===== FULL version, first 40 lines of one resume =====')
print('\n'.join(V['full'][ids[0]].split('\n')[:40]))

## 2. Score every version

Every other candidate keeps its **original** resume (as in Notebooks 8 and 8b), so "places up" is comparable. Sanity check: `flat` should score almost the same as Notebook 8b's cleaned version.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

target = evald[evald.resume_id.isin(ids)].copy()
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=100_000)
tfidf.fit(list(evald.job_text.unique()) + list(evald.resume_cut.unique()))
def tfidf_cos(jobs, resumes):
    return np.asarray(tfidf.transform(jobs).multiply(tfidf.transform(resumes)).sum(axis=1)).ravel()
evald['tfidf'] = tfidf_cos(evald.job_text, evald.resume_cut)
for s in STYLES:
    target['tfidf_' + s] = tfidf_cos(target.job_text, target.resume_id.map(V[s]))

MODELS = [('e5', 'intfloat/e5-base-v2', 'query: ', 'passage: '),
          ('bge', 'BAAI/bge-base-en-v1.5', '', ''),
          ('mpnet', 'sentence-transformers/all-mpnet-base-v2', '', '')]
res_ids = list(res_cut.index)
tok_len = {}
for col, hf_id, qp, dp in MODELS:
    m = SentenceTransformer(hf_id, device=DEVICE); m.max_seq_length = 512
    enc = lambda texts, p: m.encode([p + t for t in texts], batch_size=32, normalize_embeddings=True, show_progress_bar=False)
    je = dict(zip(jobs_u.job_id, enc(jobs_u.job_text.tolist(), qp)))
    ro = dict(zip(res_ids, enc(res_cut.tolist(), dp)))
    evald[col] = [float(je[j] @ ro[r]) for j, r in zip(evald.job_id, evald.resume_id)]
    for s in STYLES:
        if s in ('flat', 'full'):          # all resumes, for the "everyone formats" test in section 5
            rv = dict(zip(res_ids, enc([V[s][r] for r in res_ids], dp)))
            evald[f'{col}_all{s}'] = [float(je[j] @ rv[r]) for j, r in zip(evald.job_id, evald.resume_id)]
        else:
            rv = dict(zip(ids, enc([V[s][r] for r in ids], dp)))
        target[f'{col}_{s}'] = [float(je[j] @ rv[r]) for j, r in zip(target.job_id, target.resume_id)]
    if col == 'e5':
        tok_len = {s: np.mean([len(m.tokenizer(dp + V[s][r])['input_ids']) for r in ids]) for s in STYLES}
    del m; gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    print(col, 'done', flush=True)
print('mean E5 tokens per resume (cut at 512):', {s: round(v) for s, v in tok_len.items()})

CONTENT = ['tfidf', 'e5', 'bge', 'mpnet']
for col in CONTENT:
    target[col + '_orig'] = target.pair_key.map(evald.set_index('pair_key')[col])
    by_job = evald.groupby('job_id')[col].apply(np.array).to_dict()
    for s in STYLES:
        target[f'{col}_rank_{s}'] = [1 + int((by_job[j] > x).sum()) - int(o > x)
                                     for j, o, x in zip(target.job_id, target[col + '_orig'], target[f'{col}_{s}'])]

ref = nb8b.set_index('pair_key')
for col in CONTENT:
    d = (target.set_index('pair_key')[f'{col}_flat'] - ref[f'{col}_clean']).abs()
    print(f'{col:6} flat vs Notebook 8b cleaned: mean difference {d.mean():.4f}, max {d.max():.4f}')
print('scored (a mean difference above 0.01 means the versions drifted)')

## 3. Results: what does each formatting step do?

Steps, each on the same resumes: **colons and line breaks** (flat → lines), **bold headings** (lines → bold), **bullets** (lines → bullets), **bold + bullets** (lines → full), and **all formatting** (flat → full).

Compare **all formatting** for E5 with Notebook 8b's markdown effect on the AI text (+0.66 SD) and its total polish effect (+0.74 SD). TF-IDF ignores symbols, so its markdown steps should be about zero (a built-in sanity check).

In [ ]:
def holm(p):
    p = np.asarray(p, dtype=float); order = np.argsort(p); k = len(p); adj = np.empty(k); running = 0
    for rank, i in enumerate(order):
        running = max(running, (k - rank) * p[i]); adj[i] = min(1, running)
    return adj

def cluster_ci(d, groups, n=2000):
    df = pd.DataFrame({'d': d, 'g': groups}); sums = df.groupby('g').d.agg(['sum', 'count'])
    rng = np.random.default_rng(SEED); ids_ = np.arange(len(sums)); out = []
    for _ in range(n):
        b = sums.iloc[rng.choice(ids_, len(ids_))]
        out.append(b['sum'].sum() / b['count'].sum())
    return np.percentile(out, [2.5, 97.5])

STEPS = [('colons + line breaks', 'flat', 'lines'), ('bold headings', 'lines', 'bold'), ('bullets', 'lines', 'bullets'),
         ('bold + bullets', 'lines', 'full'), ('all formatting', 'flat', 'full')]
rows = []
for step, a, b in STEPS:
    for col in CONTENT:
        sd = evald[col].std(); d = (target[f'{col}_{b}'] - target[f'{col}_{a}']).to_numpy()
        lo, hi = cluster_ci(d, target.resume_id.to_numpy())
        rows.append({'step': step, 'method': col, 'n': len(d), 'change in SDs': round(d.mean() / sd, 3),
                     'CI (SDs)': f'[{lo / sd:.3f}, {hi / sd:.3f}]', 'share up': round((d > 0).mean(), 3),
                     'mean places up': round((target[f'{col}_rank_{a}'] - target[f'{col}_rank_{b}']).mean(), 2),
                     'share reaching top 5': round(((target[f'{col}_rank_{a}'] > 5) & (target[f'{col}_rank_{b}'] <= 5)).mean(), 3),
                     'p': wilcoxon(d).pvalue if np.any(d != 0) else 1.0})
steps = pd.DataFrame(rows)
steps['p (Holm)'] = steps.groupby('step').p.transform(lambda s: holm(s.to_numpy()))
pd.set_option('display.width', 250)
print(steps.pivot(index='method', columns='step', values='change in SDs')[[s for s, _, _ in STEPS]], '\n')

# Is the all-formatting boost the same for fit and no-fit pairs? (per resume: fit change minus no-fit change)
for col in CONTENT:
    d = target.assign(d=target[f'{col}_full'] - target[f'{col}_flat'], fit=target.relevance > 0)
    per = d.groupby(['resume_id', 'fit']).d.mean().unstack().dropna()
    gap = (per[True] - per[False]) / evald[col].std()
    print(f'{col:6} all formatting, fit minus no-fit: {gap.mean():+.3f} SDs over {len(gap)} resumes (p = {wilcoxon(gap).pvalue:.3g})')
steps

## 4. Why? Markdown in the job descriptions

If job posts contain `*` bullets or `**` bold, E5 may treat these symbols as shared content. We count them per job and check whether the boost is bigger for jobs that contain more of them, both for the added markdown here and for the AI text in Notebook 8b.

In [ ]:
jm = pd.DataFrame({'job_id': jobs_u.job_id,
                   'stars': jobs_u.job_text.str.count(r'\*'),
                   'bold': jobs_u.job_text.str.count(r'\*\*'),
                   'bullet_chars': jobs_u.job_text.str.count('[•●▪·]'),
                   'hashes': jobs_u.job_text.str.count('#'),
                   'newlines': jobs_u.job_text.str.count('\n')}).set_index('job_id')
print('share of jobs containing each:', (jm > 0).mean().round(3).to_dict())
print('mean count per job:', jm.mean().round(1).to_dict(), '\n')

target['job_stars'] = target.job_id.map(jm.stars)
ref8b = nb8b.set_index('pair_key')
target['e5_ai_md'] = target.pair_key.map(ref8b.e5_pol - ref8b.e5_plain)
for name, d in [('E5 added markdown (flat -> full)', target.e5_full - target.e5_flat),
                ('E5 AI markdown (8b: plain -> polished)', target.e5_ai_md),
                ('BGE added markdown (flat -> full)', target.bge_full - target.bge_flat)]:
    rho, p = spearmanr(target.job_stars, d)
    has, no = d[target.job_stars > 0], d[target.job_stars == 0]
    print(f'{name:40} rho with job stars = {rho:+.3f} (p = {p:.3g}) | mean gain: jobs with stars {has.mean():+.4f} '
          f'(n={len(has)}), without {no.mean():+.4f} (n={len(no)})')

## 5. If every resume had markdown, would ranking change?

All candidates of every job get the `full` version, compared with all `flat`. NDCG@10 per job (the Notebook 3b definition), paired permutation test. If nothing changes, markdown only matters when **some** candidates use it and others don't, which is the unfair case.

In [ ]:
def dcg(rels):
    rels = np.asarray(rels, dtype=float)
    return np.sum((2 ** rels - 1) / np.log2(np.arange(2, len(rels) + 2)))
def ndcg_at_k(r, k):
    ideal = dcg(sorted(r, reverse=True)[:k])
    return dcg(r[:k]) / ideal if ideal > 0 else 0.0
def paired_permutation_p(a, b, n=10000, seed=SEED):
    rng = np.random.default_rng(seed); d = np.asarray(a) - np.asarray(b)
    signs = rng.choice([-1, 1], size=(n, len(d)))
    return float((np.abs((signs * d).mean(axis=1)) >= abs(d.mean())).mean())

rows = []
for col in ['e5', 'bge', 'mpnet']:
    per = {}
    for v in ('allflat', 'allfull'):
        rng = np.random.default_rng(SEED); vals = []
        for jid, grp in evald.groupby('job_id', sort=True):
            s = grp[f'{col}_{v}'].to_numpy(); rel = grp.relevance.to_numpy()
            vals.append(ndcg_at_k(rel[np.lexsort((rng.random(len(s)), -s))].tolist(), 10))
        per[v] = np.array(vals)
    rows.append({'method': col, 'jobs': len(per['allflat']), 'NDCG@10 all flat': round(per['allflat'].mean(), 4),
                 'NDCG@10 all markdown': round(per['allfull'].mean(), 4),
                 'diff': round((per['allfull'] - per['allflat']).mean(), 4), 'p': paired_permutation_p(per['allfull'], per['allflat'])})
everyone = pd.DataFrame(rows)
everyone['p (Holm)'] = holm(everyone.p.to_numpy())
everyone

## 6. Save

In [ ]:
steps.to_csv(f'{RESULTS}/08c_markdown_steps.csv', index=False)
everyone.to_csv(f'{RESULTS}/08c_everyone_ndcg.csv', index=False)
jm.to_csv(f'{RESULTS}/08c_job_markdown_counts.csv')
keep = ['job_id', 'resume_id', 'pair_key', 'relevance', 'job_stars'] + \
       [f'{c}_{s}' for c in CONTENT for s in ['orig'] + STYLES] + [f'{c}_rank_{s}' for c in CONTENT for s in STYLES]
target[keep].to_parquet(f'{RESULTS}/08c_pair_scores.parquet', index=False)
print('saved. Next: download this notebook (File > Download > .ipynb).')